In [1]:
# MONDAY test
# an even smaller time period.
# JJ is having trouble keeping track of the order NAME.
# this version is trying the clip before the order is placed.

import json
import os
import pathlib
import time
import geopandas as gpd
import pandas as pd

import requests
from requests.auth import HTTPBasicAuth
from planet import Session, DataClient, OrdersClient, Auth, Planet

crs = "EPSG:4326"
sb_bbox = [0, 0, 0, 0]

data_api_url = "https://api.planet.com/data/v1"
orders_api_url = 'https://api.planet.com/compute/ops/orders/v2' 



Set api key and authorize

In [2]:
# if your Planet API Key is not set as an environment variable, you can paste it below
# jonjab's API key:
if os.environ.get('PL_API_KEY') == None:
    os.environ["PL_API_KEY"] = ''
    
planet_key = os.environ.get("PL_API_KEY")

# authentication
auth = HTTPBasicAuth(planet_key, "")
response = requests.get(data_api_url, auth=auth)
print(response)

# make function for pagenation
def p(data):
    print(json.dumps(data, indent = 2))

<Response [200]>


init session

In [3]:
# setup
session = requests.Session()

# authenticate
session.auth = (planet_key, "")

res = session.get(data_api_url)

where is your polygon geojson file?

In [4]:
! dir

 Volume in drive D is Storage 1
 Volume Serial Number is 8235-0EFB

 Directory of d:\Sequoia

04/07/2025  09:38 AM    <DIR>          .
04/07/2025  09:38 AM    <DIR>          ..
06/27/2024  03:59 PM               370 greater_UCSB-campus-aoi.geojson
04/04/2025  02:42 PM            62,243 planet_orders_jj.ipynb
04/07/2025  09:24 AM            25,539 planet_orders_jj_2.ipynb
04/07/2025  09:38 AM            26,407 planet_orders_jj_3.ipynb
04/04/2025  10:34 AM            15,490 planet_orders_test.ipynb
03/18/2025  11:52 AM    11,706,562,968 Sequoia-20240909_to_20250101_psscene_analytic_sr_udm2.zip
03/14/2025  12:24 PM               269 sequoia-975sqkmAOI.geojson
03/18/2025  01:37 PM               269 sequoia-AOI.geojson
03/17/2025  02:06 PM     5,335,508,360 sequoia-to-20180716_normalized_psscene_analytic_sr_udm2.zip
03/14/2025  02:58 PM     5,457,670,732 sequoia-to-20180716_psscene_analytic_sr_udm2.zip
03/17/2025  02:57 PM     6,389,610,606 sequoia-to-20180923_psscene_analytic_sr_udm2.zip
0

In [5]:
# sb_county = gpd.read_file("sequoia_975sqkmAOI.geojson")
sb_county = gpd.read_file("greater_UCSB-campus-aoi.geojson")
type(sb_county)

geopandas.geodataframe.GeoDataFrame

massage that geojson

In [7]:
sb_json = (
    dict( # convert to dict because first index outputs as list
        sb_county.pipe(gpd.GeoDataFrame)
        .to_geo_dict() # convert polygon to geojson format
        ["features"][0] # select first and only feature
        )
        ["geometry"] # select geometry attribute
    )

len(sb_json["coordinates"][0])

10

In [8]:
# output the AOI
# sb_county.simplify(0.03).to_frame(name="geometry").pipe(gpd.GeoDataFrame).boundary.iloc[0]
sb_json

# can we get a basemap?
# sb_json.to_frame(name="geometry")

{'type': 'Polygon',
 'coordinates': (((-119.754963, 34.458006),
   (-119.754115, 34.403032),
   (-119.831228, 34.413051),
   (-119.841397, 34.400236),
   (-119.87953, 34.401634),
   (-119.896478, 34.414915),
   (-119.912296, 34.421206),
   (-119.925854, 34.429825),
   (-119.925572, 34.456609),
   (-119.754963, 34.458006)),)}

Set geometry, date, and cloud filters
* alas, there's no coverage % filter in the search api. so there's a lot of work left to do.
* try kepler: https://planet-sdk-for-python-v2.readthedocs.io/en/latest/cli/cli-tips-tricks/
* and visualizing results 

In [9]:
# set geometry filter
geometry_filter = {
    "type": "GeometryFilter", 
    "field_name": "geometry", 
    "config": sb_json
}

# # set date filter
# date_range_filter = {
#     "type": "DateRangeFilter", 
#     "field_name": "acquired",  
#     "config": { 
#         "gte": "2025-01-01T00:00:00.000Z",
#         "lt":  "YYYY-MM-DDT00:00.000Z"
#     }
# }

# DATE RANGE FILTER
# 
date_range_filter = {
    "type": "DateRangeFilter", 
    "field_name": "acquired",  
    "config": { 
         "gte": "2025-01-15T00:00:00.000Z",
         "lt":  "2025-01-20T00:00:00.000Z"
    }
}

# # set cloud cover filter
# cloud_cover_filter = {
#     "type": "RangeFilter", 
#     "field_name": "cloud_cover", 
#     "config": {
#         "lt": 0.04
#     }
# }

# Set cloud filter
cloud_cover_filter = {
    "type": "RangeFilter", 
    "field_name": "cloud_cover", 
    "config": {
        "lt": 0.1
    }
}

# combine filters
combined_filters = {
    "type": "AndFilter", 
    "config": [geometry_filter, date_range_filter, cloud_cover_filter]
}

Run quick search based on filters

In [10]:
item_types = ["PSScene"]

search_request = {
    "item_types": item_types,
    "filter": combined_filters
}

search_result = \
    requests.post(
        "https://api.planet.com/data/v1/quick-search",
        auth = HTTPBasicAuth(planet_key, ''), 
        json = search_request
    )

print(search_result)

<Response [200]>


Print results scene ids

In [11]:
first_ids = [feature['id'] for feature in search_result.json()["features"]]
first_ids

['20250117_185707_21_24f8',
 '20250117_185852_59_2513',
 '20250115_190156_16_24cb',
 '20250115_190158_33_24cb',
 '20250115_185758_40_24d5',
 '20250115_185756_21_24d5']

Print all scene ids

In [12]:
all_ids = []
all_ids.extend(first_ids)
loop_trigger = len(first_ids)

id_json = search_result.json()

while loop_trigger == 250:
    next_url = id_json["_links"]["_next"]

    next_250 = session.get(next_url)

    id_json = next_250.json()

    features = next_250.json()["features"]


    id_list = []

    for f in features:
        
        # print id for each feature
        id_str = f["id"]
        id_list.extend([id_str])

    all_ids.extend(id_list)

    print(f"num ids: {len(all_ids)}")

    loop_trigger = len(id_list)

In [ ]:
# not_str_cnt = 0
# for id in all_ids:
#     if isinstance(id, str) != True:
#         print(f"{id} not string!!")
#         not_str_cnt = not_str_cnt + 1

# print(f"not string count: {not_str_cnt}")

In [13]:
# How many?
len(all_ids)

6

In [14]:
all_ids

['20250117_185707_21_24f8',
 '20250117_185852_59_2513',
 '20250115_190156_16_24cb',
 '20250115_190158_33_24cb',
 '20250115_185758_40_24d5',
 '20250115_185756_21_24d5']

### Requests example

Now let's use `requests` to communicate with the orders v2 API. First, we will check our orders list to make sure authentication and communication is working as expected.

We want to get a response code of `200` from this API call. To troubleshoot other response codes, see the [List Orders](https://developers.planet.com/apis/orders/reference/#tag/Orders/operation/listOrders) API reference.

In [15]:
auth = HTTPBasicAuth(planet_key, "")
order_response = requests.get(orders_api_url, auth=auth)
print(order_response)

<Response [200]>


Print previous orders
this has stopped working

In [23]:
# orders = order_response.json()['orders']
# [r['name'] for r in orders[:15]]
previous_orders = order_response.json()['orders']

In [24]:
type(previous_orders)
len(previous_orders)

20

In [26]:
previous_orders.name

AttributeError: 'list' object has no attribute 'name'

# Make order request: CHANGE THE NAME here

In [27]:
# set content type to json
headers = {"content-type": "application/json"}

# init order parameters
product = [
    { 
        "item_ids": all_ids, 
        "item_type": "PSScene", 
        "product_bundle": "analytic_8b_sr_udm2", 
    }
]

order_request = {
    "name": "monday_test_clip_try_3", 
    "products": product, 
    "delivery": {"single_archive": True, "archive_type": "zip"}
}

In [28]:
def place_order(request, auth):

    # make order request
    response = requests.post(
        orders_api_url, 
        data = json.dumps(request), 
        auth = auth, 
        headers = headers
        )
    print(response.json())

    # get ids of scenes
    order_id = response.json()["id"]
    print(order_id)

    # construct the url of our order
    order_url = orders_api_url + '/' + order_id
    
    return order_url

In [29]:
# init clip parameters
clip =  {
    "clip": {
        "aoi": sb_json
    }
}

clip_request = { 
    "name": "", 
    "products": product, 
    "tools": [clip], 
    "delivery": {"single_archive": True, "archive_type": "zip"}
}

In [ ]:
# place our order
# this really is the line that places the order
# Jon REALLY thinks that clip needs to be done before here. 
# order_url = place_order(order_request, auth)
# it's not catching the name now that I am passing the clip request. 
clip_request
order_url = place_order(clip_request, auth)

{'_links': {'_self': 'https://api.planet.com/compute/ops/orders/v2/49fdfeb8-8538-42b2-8187-1e0c596df60c'}, 'created_on': '2025-04-07T17:01:16.544703Z', 'delivery': {'archive_filename': 'output.zip', 'archive_type': 'zip', 'single_archive': True}, 'error_hints': [], 'id': '49fdfeb8-8538-42b2-8187-1e0c596df60c', 'last_message': 'Preparing order', 'last_modified': '2025-04-07T17:01:16.544703Z', 'products': [{'item_ids': ['20250117_185707_21_24f8', '20250117_185852_59_2513', '20250115_190156_16_24cb', '20250115_190158_33_24cb', '20250115_185758_40_24d5', '20250115_185756_21_24d5'], 'item_type': 'PSScene', 'product_bundle': 'analytic_8b_sr_udm2'}], 'state': 'queued', 'tools': [{'clip': {'aoi': {'coordinates': [[[-119.754963, 34.458006], [-119.754115, 34.403032], [-119.831228, 34.413051], [-119.841397, 34.400236], [-119.87953, 34.401634], [-119.896478, 34.414915], [-119.912296, 34.421206], [-119.925854, 34.429825], [-119.925572, 34.456609], [-119.754963, 34.458006]]], 'type': 'Polygon'}}}]}


In [34]:
# order_url
# can we output the order name here too?

# this is me trying my clipped order. 
# clip_order_url = place_order(clip_request, auth)

### Poll for order success
The timing on this loop has been shortened so that you can see it running better
You need to keep running this poll until your order is ready.
The bigger the order, the longer that takes.

In [37]:
def poll_for_success(order_url, auth, num_loops = 4): 
    i = 0
    while(i < num_loops): 

        # iterate
        i += 1

        # get order request
        r = requests.get(order_url, auth = auth)
        response = r.json()

        # grab current state
        state = response["orders"][0]["state"]
        print(state)
        print(order_request)

        # compare it to a variety of end states and print it
        end_states = ["success", "failed", "partial"]
        if state in end_states:
            print(f"End State: {state}")
            break

        # wait 3 secs
        time.sleep(3)

poll_for_success(orders_api_url, auth)

running
{'name': 'monday_test_clip_try_3', 'products': [{'item_ids': ['20250117_185707_21_24f8', '20250117_185852_59_2513', '20250115_190156_16_24cb', '20250115_190158_33_24cb', '20250115_185758_40_24d5', '20250115_185756_21_24d5'], 'item_type': 'PSScene', 'product_bundle': 'analytic_8b_sr_udm2'}], 'delivery': {'single_archive': True, 'archive_type': 'zip'}}
running
{'name': 'monday_test_clip_try_3', 'products': [{'item_ids': ['20250117_185707_21_24f8', '20250117_185852_59_2513', '20250115_190156_16_24cb', '20250115_190158_33_24cb', '20250115_185758_40_24d5', '20250115_185756_21_24d5'], 'item_type': 'PSScene', 'product_bundle': 'analytic_8b_sr_udm2'}], 'delivery': {'single_archive': True, 'archive_type': 'zip'}}
running
{'name': 'monday_test_clip_try_3', 'products': [{'item_ids': ['20250117_185707_21_24f8', '20250117_185852_59_2513', '20250115_190156_16_24cb', '20250115_190158_33_24cb', '20250115_185758_40_24d5', '20250115_185756_21_24d5'], 'item_type': 'PSScene', 'product_bundle': 'an

### Clip to AOI Parameters: sb_json    
# ok: Jon's script runs up to here.
Joshua was clipping AFTER he has already submitted the order.
In this example, I clip before I submit the order and what gets assembled
is clipped just fine.

##### Place order and check for order success
 but the order is already placed and ready!!!!  

In [ ]:
# this isn't a clip error
clip_order_url = place_order(clip_request, auth)

In [ ]:
# This isn't a clip error. 
poll_for_success(clip_order_url, auth)

View results of order

In [ ]:
r = requests.get(orders_api_url, auth = auth)
clip_response = r.json()
# order_results = clip_response["orders"][0]["state"]
order_results

### download

In [ ]:
# download results locally and individually
# running this with order_url throws all sorts of errors
def download_results(results, overwrite=False):
    results_urls = [r['location'] for r in results]
    results_names = [r['name'] for r in results]
    print('{} items to download'.format(len(results_urls)))
    
    for url, name in zip(results_urls, results_names):
        path = pathlib.Path(os.path.join('..', '..', '..', 'data', name))
        
        if overwrite or not path.exists():
            print('downloading {} to {}'.format(name, path))
            r = requests.get(url, allow_redirects=True)
            path.parent.mkdir(parents=True, exist_ok=True)
            open(path, 'wb').write(r.content)
        else:
            print('{} already exists, skipping {}'.format(path, name))

In [ ]:
# order_url
# download_results(order_results)
download_results(order_url)

In [ ]:
# cancel an order
order_id = "641422a1-e8d9-4641-9df9-d85be29cd44f"


def cancel_order(order_id):
    cancelled_order = pl.orders.cancel_order(order_id)
    return cancelled_order